In [17]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded))

Saving Transformer_Math_and_RAG_Explained.pdf to Transformer_Math_and_RAG_Explained.pdf


In [19]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 24.4 MB/s eta 0:00:00


In [1]:
!pip install -q \
sentence-transformers \
chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [21]:
import fitz
import re

document = fitz.open(pdf_path)

text = ""

for page in document:
    text += page.get_text()

clean_text = re.sub(r"\s+", " ", text).strip()

In [22]:
CHUNK_SIZE = 500
OVERLAP = 100

chunks = []

start = 0

while start < len(clean_text):

    end = start + CHUNK_SIZE

    chunks.append(clean_text[start:end])

    start += CHUNK_SIZE - OVERLAP

print(len(chunks))

25


In [2]:
from sentence_transformers import SentenceTransformer

import chromadb

import torch

In [3]:
print("Torch :", torch.__version__)

print("Chroma Installed")

print("Sentence Transformers Ready")

Torch : 2.11.0+cpu
Chroma Installed
Sentence Transformers Ready


In [4]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

print(MODEL_NAME)

BAAI/bge-small-en-v1.5


In [5]:
embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Embedding Model Ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Ready


In [6]:
print(type(embedding_model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [23]:
chunk_embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
print(type(chunk_embeddings))
print()
print(chunk_embeddings.shape)

<class 'numpy.ndarray'>

(25, 384)


In [25]:
print(chunk_embeddings[0].shape)

print()

print(chunk_embeddings[0][:25])

(384,)

[-0.04280171  0.04787553 -0.02658868  0.01902101 -0.02734322  0.0301138
  0.00501574  0.0133131   0.02314126 -0.00140985 -0.00220745 -0.03725723
  0.04774582  0.0022331   0.06591262  0.0377936  -0.01097895  0.00945553
 -0.02215777 -0.06964059  0.04830178  0.00660054 -0.00688744  0.01563664
 -0.01088481]


In [26]:
from sentence_transformers import util

print("Chunk 0 ↔ Chunk 1")

print(
    util.cos_sim(
        chunk_embeddings[0],
        chunk_embeddings[1]
    )
)

print()

print("Chunk 0 ↔ Chunk 20")

print(
    util.cos_sim(
        chunk_embeddings[0],
        chunk_embeddings[20]
    )
)


Chunk 0 ↔ Chunk 1
tensor([[0.7870]])

Chunk 0 ↔ Chunk 20
tensor([[0.7687]])


In [27]:
print("Embedding Matrix Shape")

print(chunk_embeddings.shape)

print()

print("Total Float Values")

print(chunk_embeddings.size)

print()

print("Approx Memory")

print(chunk_embeddings.nbytes / 1024, "KB")

Embedding Matrix Shape
(25, 384)

Total Float Values
9600

Approx Memory
37.5 KB


In [28]:
client = chromadb.Client()

print(client)

In [29]:
collection = client.create_collection(
    name="enterprise_documents"
)

print(collection.name)

enterprise_documents


In [30]:
ids = []

for i in range(len(chunks)):
    ids.append(f"chunk_{i}")

print(ids[:5])

['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4']


In [31]:
collection.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)

print("Knowledge Stored Successfully!")

Knowledge Stored Successfully!


In [32]:
print(collection.count())

25


In [33]:
query = "How do transformers convert words into vectors?"

print(query)

How do transformers convert words into vectors?


In [34]:
query_embedding = embedding_model.encode(query)

print(query_embedding.shape)

(384,)


In [35]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

In [36]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [37]:
print("=" * 60)

print(results["documents"][0][0])

print("=" * 60)

print(results["documents"][0][1])

print("=" * 60)

print(results["documents"][0][2])

learned computation involved. Step 2 — Integers Become Vectors (Embeddings) In plain terms A single number can't describe what a word means — so instead, each token ID is turned into a long list of numbers (a vector). Imagine coordinates on a map, except instead of just latitude and longitude, there are hundreds of coordinates, each capturing a tiny shade of meaning: how "animal-like", how "large", how "emotional" a concept is. No single number matters on its own — together, they define meaning.
ddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-vector — the closer they are, the more relevant that document probably is. The math similarity(q, d) = (q · d) / (||q|| × ||d||) Technical detail Both documents and the incoming query are passed throu